# Spandan — YOLOv8 Training (Colab GPU)

Runs the Phase 2 pipeline on **RDD2022 + CODEBRIM** and exports `yolov8_spandan.pt`.

**First:** Runtime → Change runtime type → **T4 GPU**.
Full RDD2022 training needs a GPU; on CPU it is impractically slow.

In [ ]:
!nvidia-smi

In [ ]:
# 1) Clone the repo (public) and enter the AI package
!git clone https://github.com/kaushalv17/spandan.git
%cd spandan/ai

In [ ]:
# 2) Install the AI package with ML extras
!pip -q install -e ".[dev,ml,service]"

In [ ]:
# 3) Prepare datasets (RDD2022 + CODEBRIM auto-download; SDNET/DeepCrack are manual).
#    Produces data/processed/spandan.yaml with the 8 unified classes.
!python -m spandan_ai.data.prepare

In [ ]:
# 4) Train YOLOv8 (Hydra CLI overrides). Raise epochs for quality; 50 is a solid demo model.
!python -m spandan_ai.train model.epochs=50 model.imgsz=640

In [ ]:
# 5) Copy the best checkpoint to the path the service auto-loads
import glob, shutil, os
best = sorted(glob.glob('runs/**/weights/best.pt', recursive=True))
assert best, 'no best.pt found — check the training run output above'
os.makedirs('weights', exist_ok=True)
shutil.copy(best[-1], 'weights/yolov8_spandan.pt')
print('saved weights/yolov8_spandan.pt from', best[-1])

In [ ]:
# 6) Download the trained weights to your machine
from google.colab import files
files.download('weights/yolov8_spandan.pt')

## Use the weights locally
Drop the downloaded `yolov8_spandan.pt` into `spandan/ai/weights/` and restart the AI service:

```powershell
cd C:\dev2\spandan\ai
uvicorn spandan_ai.service.app:app --port 8001
```

On start it logs `detector_loaded` (real weights) instead of `detector_fallback_demo`.
No code change needed — the Phase 6 loader picks up the file automatically.